# CarbonAlpha Final Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/capabl-machines/gridops/blob/round-2/notebooks/carbonalpha_final_pipeline.ipynb)

This is the final submission notebook for CarbonAlpha: a climate-aware portfolio OpenEnv where an LLM reads macro news, reasons through first-order and second-order effects, and emits a strict portfolio action.

The notebook is deliberately transparent:

- It verifies the environment and Hugging Face artifacts.
- It shows the exact SFT and GRPO lineage.
- It loads the saved metrics from the model repo.
- It exposes the 10-question macro eval set.
- It includes opt-in cells to relaunch the HF Jobs training runs.

Heavy training is run through Hugging Face Jobs rather than directly inside Colab. That is the path we actually validated, and it avoids pretending a free Colab T4 is the same as the L40S jobs used for the final artifacts.

## TL;DR Results

Best current research model:

```text
77ethers/CarbonAlpha/grpo_qwen25_7b_adapter_phase1_100_v1
```

Lineage:

1. SFT: `unsloth/Qwen2.5-7B-Instruct` plus LoRA on `sft_traces/curriculum_400_e80_m160_h160.jsonl`.
2. GRPO: 100 Phase-1 steps with plain Transformers generation, `use_vllm=False`.
3. Evaluation: 5/5 holdout validity, mean holdout regret `+0.1058`, beats equal-weight baseline on 5/5 seeds.

Important caveat:

- Qwen3-4B-Base is mechanically alive after our fixes, but its smoke run did not beat Qwen2.5 yet.
- The Qwen3 smoke used `merged_v6_aligned.jsonl`; the fair next attempt should use the 400-trace curriculum.

Training evidence and limitations are written up in `MODEL_CARD.md`, including real HF Job logs and plots generated from the 100-step GRPO run.


## 0. Configuration

Set `HF_API_TOKEN` when prompted. The model and dataset repos are private, so the token needs read access. Cells that launch training jobs are guarded by booleans and default to `False`.

In [ ]:
REPO_URL = "https://github.com/capabl-machines/gridops.git"
BRANCH = "round-2"  # change to "main" after merge, if needed

MODEL_REPO = "77ethers/CarbonAlpha"
CODE_REPO = "77ethers/CarbonAlpha-train"

SAFE_NUMERICAL_MODEL = "v6_sft_only_v2"
SFT_QWEN25_MODEL = "sft_qwen25_7b_curriculum400_v1"
BEST_GRPO_MODEL = "grpo_qwen25_7b_adapter_phase1_100_v1"
QWEN3_BASE_SMOKE = "grpo_qwen3_4b_base_smoke_v2"

SPACE_URL = "https://77ethers-carbonalpha-demo.hf.space/"
COLAB_URL = "https://colab.research.google.com/github/capabl-machines/gridops/blob/round-2/notebooks/carbonalpha_final_pipeline.ipynb"

print("Best GRPO model:", BEST_GRPO_MODEL)
print("Demo Space:", SPACE_URL)
print("Colab link:", COLAB_URL)

## 1. Install and Clone

This cell works in Colab and locally. It installs only the lightweight packages needed for review and HF Jobs launch. Training dependencies live inside the PEP 723 headers of the job scripts.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

PROJECT_DIR = Path.cwd()
if "google.colab" in sys.modules:
    PROJECT_DIR = Path("/content/gridops")
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)
else:
    # Local repo path if the notebook is opened from the checkout.
    if (Path.cwd() / "portfolio_env").exists():
        PROJECT_DIR = Path.cwd()
    else:
        PROJECT_DIR = Path("/content/gridops")
        if not PROJECT_DIR.exists():
            subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
        os.chdir(PROJECT_DIR)

print("Project dir:", Path.cwd())

%pip install -q -e .
%pip install -q "huggingface_hub>=0.34" pandas matplotlib

## 2. Authenticate and Verify Repos

We always use `HF_API_TOKEN`. Some libraries implicitly read `HF_TOKEN`, so we mirror the same value into both names inside the runtime.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi

if not os.environ.get("HF_API_TOKEN"):
    os.environ["HF_API_TOKEN"] = getpass("Paste HF_API_TOKEN with read access to the private repos: ")
os.environ["HF_TOKEN"] = os.environ["HF_API_TOKEN"]
os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_API_TOKEN"]

api = HfApi(token=os.environ["HF_API_TOKEN"])
who = api.whoami(token=os.environ["HF_API_TOKEN"])
print("Authenticated as:", who.get("name"))

for repo_id, repo_type in [(MODEL_REPO, "model"), (CODE_REPO, "dataset")]:
    info = api.repo_info(repo_id=repo_id, repo_type=repo_type, token=os.environ["HF_API_TOKEN"])
    files = api.list_repo_files(repo_id=repo_id, repo_type=repo_type, token=os.environ["HF_API_TOKEN"])
    print(f"{repo_type}:{repo_id} private={getattr(info, 'private', None)} files={len(files)}")

## 3. Environment Smoke Test

This is the OpenEnv core. The model produces one locked allocation. The environment rolls it through the path-dependent 12-quarter simulator.

In [ ]:
from portfolio_env import PortfolioEnv, PortfolioAction, r_regret, r_carbon, r_drawdown

example_action = PortfolioAction(
    weights=[0.30, 0.05, 0.35, 0.05, 0.25],
    infra_commit=0.0,
    carbon_offset_buy=0.0,
    put_hedge=0.0,
    tech_bet="status_quo",
)

env = PortfolioEnv(phase=3, seed=100)
obs = env.reset(seed=100)
print("Q0 news:", obs.news[:300], "...")

for _ in range(12):
    obs = env.step(example_action, completion="manual smoke test")
    if obs.done:
        break

traj = env.trajectory
print("Final real NAV:", round(traj.nav_real_series[-1], 4))
print("Baseline real NAV:", round(traj.baseline_nav_real_series[-1], 4))
print("Regret:", round(r_regret(traj), 4))
print("Carbon reward:", round(r_carbon(traj), 4))
print("Drawdown reward:", round(r_drawdown(traj), 4))

## 4. Artifact Ledger

These are the artifacts we actually produced. Known-good models are never overwritten; every experiment writes a new subfolder.

In [ ]:
import pandas as pd

artifacts = [
    {
        "role": "safe numerical baseline",
        "subfolder": SAFE_NUMERICAL_MODEL,
        "notes": "Qwen3-4B-Instruct SFT-only, mean holdout regret +0.034, 5/5 valid",
    },
    {
        "role": "SFT warm-start for Qwen2.5",
        "subfolder": SFT_QWEN25_MODEL,
        "notes": "Qwen2.5-7B-Instruct SFT on 400 curriculum traces",
    },
    {
        "role": "best current research model",
        "subfolder": BEST_GRPO_MODEL,
        "notes": "Qwen2.5 adapter GRPO, 100 Phase-1 steps, mean holdout regret +0.1058",
    },
    {
        "role": "Qwen3 Base smoke branch",
        "subfolder": QWEN3_BASE_SMOKE,
        "notes": "Mechanically passed smoke, but holdout did not beat Qwen2.5",
    },
]

df = pd.DataFrame(artifacts)
df

## 5. Load Saved Metrics From Hugging Face

This cell reads the actual metrics files uploaded by the training jobs.

In [ ]:
import json
from pathlib import Path
from huggingface_hub import hf_hub_download


def load_model_json(filename: str):
    path = hf_hub_download(
        repo_id=MODEL_REPO,
        repo_type="model",
        filename=filename,
        token=os.environ["HF_API_TOKEN"],
    )
    return json.loads(Path(path).read_text())

qwen25_metrics = load_model_json(f"{BEST_GRPO_MODEL}/metrics.json")
qwen3_metrics = load_model_json(f"{QWEN3_BASE_SMOKE}/smoke_metrics.json")

summary_rows = [
    {
        "model": BEST_GRPO_MODEL,
        "steps": qwen25_metrics.get("grpo_steps"),
        "valid": qwen25_metrics.get("post_grpo_holdout", {}).get("valid"),
        "total": qwen25_metrics.get("post_grpo_holdout", {}).get("total"),
        "mean_regret": qwen25_metrics.get("post_grpo_holdout", {}).get("mean_regret"),
        "beats_baseline": qwen25_metrics.get("post_grpo_holdout", {}).get("beats_baseline"),
        "smoke_gate": qwen25_metrics.get("smoke_gate_passed"),
    },
    {
        "model": QWEN3_BASE_SMOKE,
        "steps": 10,
        "valid": qwen3_metrics.get("holdout_eval", {}).get("valid"),
        "total": qwen3_metrics.get("holdout_eval", {}).get("total"),
        "mean_regret": qwen3_metrics.get("holdout_eval", {}).get("mean_regret"),
        "beats_baseline": qwen3_metrics.get("holdout_eval", {}).get("beats_baseline"),
        "smoke_gate": qwen3_metrics.get("smoke_gate_passed"),
    },
]

pd.DataFrame(summary_rows)

## 6. Manual Macro Eval Set

Holdout regret tests the simulator objective. The 10-question macro eval checks whether the model is actually reading macro news sensibly.

Known weaknesses from the first GRPO eval:

- `q02_oil_chokepoint_inflation`: underweighted OIL despite direct oil supply removal.
- `q04_ai_efficiency_paradox`: overweighted GREEN despite lower data-center power demand.

In [ ]:
import json
from pathlib import Path
import pandas as pd

cases_path = Path("evals/macro_eval_10.jsonl")
cases = [json.loads(line) for line in cases_path.read_text().splitlines() if line.strip()]

pd.DataFrame([
    {
        "id": row["id"],
        "difficulty": row["difficulty"],
        "category": row["category"],
        "question_preview": row["question"][:100] + "...",
    }
    for row in cases
])

In [ ]:
report_path = Path("evals/macro_eval_10_grpo_report.json")
if report_path.exists():
    report = json.loads(report_path.read_text())
    trained = report.get("trained", {})
    rows = []
    for case_id, result in trained.items():
        score = result.get("score", {})
        rows.append({
            "id": case_id,
            "valid_action": score.get("valid_action"),
            "closed_think": score.get("closed_think"),
            "tokens_approx": score.get("tokens_approx"),
            "weights": score.get("weights"),
            "tech_bet": score.get("tech_bet"),
            "put_hedge": score.get("put_hedge"),
        })
    display(pd.DataFrame(rows))
else:
    print("No saved eval report found. Use the optional eval job cell below.")

## 7. HF Jobs Helpers

These helpers launch and monitor the exact scripts in `scripts/`. Nothing is launched unless a `RUN_*` boolean is set to `True`.

In [ ]:
import time
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_API_TOKEN"])


def launch_uv_job(script, env, flavor="l40sx1", timeout="3h"):
    job = api.run_uv_job(
        script=script,
        flavor=flavor,
        secrets={"HF_API_TOKEN": os.environ["HF_API_TOKEN"]},
        env=env,
        timeout=timeout,
        token=os.environ["HF_API_TOKEN"],
    )
    print(job.id)
    print(job.url)
    return job


def tail_job(job_id, max_polls=60, sleep_s=20):
    seen = 0
    for i in range(max_polls):
        info = api.inspect_job(job_id=job_id, token=os.environ["HF_API_TOKEN"])
        print(f"poll {i}: {getattr(info, 'status', None)}")
        logs = list(api.fetch_job_logs(job_id=job_id, token=os.environ["HF_API_TOKEN"]))
        new = logs[seen:]
        seen = len(logs)
        for line in new[-80:]:
            text = line.rstrip()
            if any(key in text for key in [
                "Pre-GRPO sanity", "Generation sanity", "{'loss'", "GRPO smoke done",
                "Post-GRPO holdout", "Holdout eval", "Smoke gate", "Uploaded artifacts",
                "Traceback", "CUDA out of memory", "ValueError",
            ]):
                print(text)
        joined = "".join(logs[-120:])
        if any(done in joined for done in ["Uploaded artifacts", "Smoke gate failed", "Traceback", "CUDA out of memory"]):
            break
        time.sleep(sleep_s)

## 8. Optional: Rerun the 10-Question Eval

This launches `scripts/hf_compare_qwen25.py` through `scripts/launch_macro_eval.py`, comparing base Qwen2.5 against the selected adapter on the exact 10 questions.

In [ ]:
RUN_MACRO_EVAL_JOB = False

if RUN_MACRO_EVAL_JOB:
    subprocess.run(
        [sys.executable, "scripts/launch_macro_eval.py", "--adapter-subdir", BEST_GRPO_MODEL],
        check=True,
    )
else:
    print("Skipped. Set RUN_MACRO_EVAL_JOB=True to relaunch the eval job.")

## 9. Optional: Relaunch Best Qwen2.5 GRPO Run

This is the best validated research path. It starts from the 400-trace SFT adapter and runs Phase-1 GRPO with plain Transformers generation (`use_vllm=False`).

In [ ]:
RUN_QWEN25_GRPO_100 = False

if RUN_QWEN25_GRPO_100:
    job = launch_uv_job(
        script="scripts/hf_grpo_qwen25_adapter.py",
        env={
            "CARBON_ALPHA_GRPO_STEPS": "100",
            "CARBON_ALPHA_GRPO_PROMPTS": "128",
            "CARBON_ALPHA_GRPO_NUM_GENERATIONS": "2",
            "CARBON_ALPHA_GRPO_BATCH": "2",
            "CARBON_ALPHA_GRPO_RUN_LABEL": "grpo_qwen25_7b_adapter_phase1_100_rerun_v1",
            "CARBON_ALPHA_SFT_SUBFOLDER": SFT_QWEN25_MODEL,
        },
        timeout="3h",
    )
else:
    print("Skipped. Set RUN_QWEN25_GRPO_100=True to launch a fresh isolated rerun.")

## 10. Optional: Qwen3-4B-Base Follow-Up With 400 Traces

The Qwen3 Base smoke proved the mechanics are alive, but it used the smaller merged trace set and did not beat Qwen2.5. This is the fairer follow-up: SFT warm-start on the 400-trace curriculum, then a short isolated GRPO smoke.

In [ ]:
RUN_QWEN3_BASE_CURRICULUM_SMOKE = False

if RUN_QWEN3_BASE_CURRICULUM_SMOKE:
    job = launch_uv_job(
        script="scripts/hf_grpo_qwen3_base.py",
        env={
            "CARBON_ALPHA_GRPO_RUN_LABEL": "grpo_qwen3_4b_base_curriculum400_smoke_v1",
            "CARBON_ALPHA_BASE_MODEL": "unsloth/Qwen3-4B-Base",
            "CARBON_ALPHA_TRACES": "sft_traces/curriculum_400_e80_m160_h160.jsonl",
            "CARBON_ALPHA_SFT_STEPS": "220",
            "CARBON_ALPHA_GRPO_MAX_STEPS": "10",
            "CARBON_ALPHA_GRPO_PROMPTS": "40",
            "CARBON_ALPHA_GRPO_NUM_GENERATIONS": "2",
            "CARBON_ALPHA_GRPO_BATCH_SIZE": "2",
            "CARBON_ALPHA_MAX_SEQ_LEN": "2048",
            "CARBON_ALPHA_LORA_RANK": "32",
            "CARBON_ALPHA_GPU_MEMORY_UTILIZATION": "0.85",
        },
        timeout="3h",
    )
else:
    print("Skipped. Set RUN_QWEN3_BASE_CURRICULUM_SMOKE=True to test this branch.")

## 11. Pipeline Summary

Final training decision:

- Ship/keep `v6_sft_only_v2` as the safe numerical baseline.
- Use `sft_qwen25_7b_curriculum400_v1` as the strong SFT warm-start.
- Treat `grpo_qwen25_7b_adapter_phase1_100_v1` as the current best research model.
- Keep Qwen3 Base as an isolated research branch until it passes the 400-trace curriculum smoke and manual macro eval.

Things to note:

- Every model writes a new Hugging Face subfolder.
- The smoke gate checks length, parse validity, gradient norm, and reward variance.
- Holdout seeds are fixed and not used in training.
- Manual macro eval catches human-facing reasoning failures that regret alone misses.